### Importing The Required Libraries 

In [1]:
from typing import TypedDict, Annotated 
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain.tools import tool 
from langchain_openai import ChatOpenAI 

C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Defining AgentState

In [2]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

### Defining The Tool 

In [3]:
@tool
def inspect_order(order_id:str):
    """ This tool looks for shipping status of the order given order id of the customer """
    db={"ODR_1":"Shipped on 8 July", "ODR_2":"Ready to ship","ODR_3":"Will Deliver today"}
    return db.get(order_id,"No such  order")

In [4]:
tools=[inspect_order]

### Loading The LLM 

In [5]:
llm=ChatOpenAI(model="gpt-3.5-turbo",temperature=0)

In [6]:
llm=llm.bind_tools(tools)

### Defining The Nodes 

In [7]:
def call_model(state:AgentState):
    print("LLM is Reasoning")
    response=model.invoke(state["messages"])
    return {messages:"response"}

### Defining The Router 

In [8]:
def should_continue(state:AgentState):
    ai_resp=state["messages"][-1]
    if ai_resp.tool_calls:
        print("AI requested a tool call,routing to the tool")
        return "tools"
    else:
        print("AI has finished answering routing to the end")
        return END